# Churn distribution and cohort analysis

**Objective.** Reconstruct the fictional Fieldwise churn experiment, explain the original zero-positive training period and assess temporal and group-selection differences before future evaluation design.

**Grain and policy.** Lifecycle events are at episode grain. Outcome candidates are episode-snapshot observations. The existing synthetic policy requires 90 days of prior history and a 30-day outcome horizon. A positive requires confirmed termination and timely ascertainment. A negative requires attested complete follow-up. The analysis reconstructs both datasets locally with a fresh Registry and preserves censored, pending and ineligible categories. No model is fitted.

**Decision question.** Is there enough observable history and event support to evaluate a churn prediction task over time? The exploratory unit is an episode at a snapshot observation time. Account identifiers remain relevant for grouping and are not churn labels.

| Outcome state | Operational meaning | Denominator used below |
|---|---|---|
| Confirmed positive | Authoritative episode termination took effect after observation and within 30 days, with evidence available by ascertainment. | Eligible binary observations, or all candidates where explicitly shown |
| Confirmed negative | Complete and attested follow-up through the outcome horizon with no qualifying termination. | Eligible binary observations, or all candidates where explicitly shown |
| Right-censored | Follow-up ends before the full horizon is observed. | All candidates |
| Pending ascertainment | Evidence or follow-up is insufficient to confirm a label under the experiment policy. | All candidates |
| Ineligible | The observation fails eligibility checks. Here the observed reason is insufficient historical coverage. | All candidates |

The [ground-truth specification](../../docs/04_data/synthetic_churn_ground_truth_and_data_quality.md) describes the governance boundary. The existing Phase 04 helper reconstructs the fictional experiment with explicit simulation follow-up through March 2024. Observation time governs available predictors. The later ascertainment time governs whether the outcome can be known.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython import get_ipython
from IPython.display import display

# Resolve the repository before importing the local src-layout package.
project = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "src" / "nowa_customer_intelligence").is_dir()
    ),
    None,
)
if project is None:
    raise RuntimeError(
        "Start the Jupyter kernel from the Nowa Customer Intelligence repository "
        "root or one of its subdirectories."
    )

source_dir = str(project / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

get_ipython().run_line_magic("matplotlib", "inline")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (10, 4), "axes.titleweight": "bold"})

## Lifecycle events and observation eligibility

**Questions.** When do authoritative terminations occur, and how many snapshots can support a binary label? Calendar event counts describe distinct episode terminations. Snapshot labels describe repeated observations. A raw termination fraction by start cohort would mix different follow-up durations and is not interpreted as a risk estimate.


In [ ]:
from nowa_customer_intelligence.data.eda_inputs import (
    REFERENCE_CUTOFF,
    fieldwise_reference,
    reference_outcomes,
)
from nowa_customer_intelligence.data.synthetic.lifecycle import LifecycleKind

with fieldwise_reference() as (run, export):
    data = reference_outcomes(run, export)
    lifecycle = pd.DataFrame(
        [
            {
                "episode": row.episode.subscription_id,
                "established": row.episode.start_date,
                "reactivation": row.prior_subscription_id is not None,
                "terminated": next(
                    (
                        fact.effective_at.date()
                        for fact in row.facts
                        if fact.kind is LifecycleKind.TERMINATED
                    ),
                    None,
                ),
            }
            for row in run.population.episodes
        ]
    )
lifecycle[["established", "terminated"]] = lifecycle[["established", "terminated"]].apply(
    pd.to_datetime
)
lifecycle["establishment quarter"] = lifecycle["established"].dt.to_period("Q")
lifecycle["observed duration (days)"] = (
    lifecycle["terminated"].fillna(pd.Timestamp(REFERENCE_CUTOFF.date())) - lifecycle["established"]
).dt.days
terminated = lifecycle.loc[lifecycle["terminated"].notna()]
coverage = lifecycle.groupby("establishment quarter").agg(
    episodes=("episode", "size"),
    terminations=("terminated", "count"),
    reactivations=("reactivation", "sum"),
)
display(coverage)
display(
    lifecycle.groupby(lifecycle["terminated"].notna())["observed duration (days)"]
    .describe()
    .round(1)
)
for unit in ("M", "Q", "Y"):
    display(
        terminated.groupby(terminated["terminated"].dt.to_period(unit))
        .size()
        .rename("terminations")
    )
outcome_counts = pd.Series(data.counts, name="observations")
outcome_summary = outcome_counts.to_frame()
outcome_summary["share of all candidates (%)"] = outcome_counts.div(outcome_counts.sum()).mul(100)
outcome_summary["share of eligible labels (%)"] = (
    outcome_counts.loc[["confirmed_positive", "confirmed_negative"]]
    .div(len(data.labelled))
    .mul(100)
)
display(outcome_summary.round(2), pd.Series(data.reasons, name="reason counts"))
positive_total = sum(row.outcome.binary_target == 1 for row in data.labelled)
print(
    f"Eligible prevalence: {positive_total:,}/{len(data.labelled):,} = "
    f"{positive_total / len(data.labelled):.2%}"
)

In [ ]:
outcomes = pd.DataFrame(
    (row.observed_at, row.outcome.state.value)
    for rows in (data.labelled, data.right_censored, data.pending, data.ineligible)
    for row in rows
).set_axis(["observed", "state"], axis=1)
outcomes["month"] = outcomes["observed"].dt.tz_localize(None).dt.to_period("M")
outcome_month = pd.crosstab(outcomes["month"], outcomes["state"]).reindex(
    columns=list(data.counts), fill_value=0
)
positive = [row for row in data.labelled if row.outcome.binary_target == 1]
positive_coverage = (
    lifecycle.set_index("episode")
    .loc[[row.subscription_id for row in positive], "establishment quarter"]
    .value_counts()
    .sort_index()
)
display(positive_coverage.rename("eligible positive observations by establishment quarter"))
first_positive = min(row.observed_at for row in positive)
first_termination = terminated["terminated"].min()
print(
    f"First positive observation {first_positive.date()}, "
    f"first termination {first_termination.date()}"
)
fig, (upper, lower) = plt.subplots(2, 1, figsize=(12, 7), layout="constrained")
outcome_month.plot.area(ax=upper, linewidth=0, colormap="Set2")
upper.set(
    title="Fieldwise candidate outcomes by observation month",
    xlabel="Observation month",
    ylabel="Episode-snapshot observations",
)
upper.legend(title="Outcome state", loc="upper left", ncols=2, fontsize=8)
event_month = terminated.groupby(terminated["terminated"].dt.to_period("M")).size()
pd.concat(
    [
        outcome_month["confirmed_positive"].rename("Positive observations"),
        event_month.rename("Authoritative terminations"),
    ],
    axis=1,
).fillna(0).plot(ax=lower, marker="o", markersize=3, color=["#23667d", "#db8351"])
lower.set(
    title="Fieldwise positive snapshots and terminations", xlabel="Calendar month", ylabel="Count"
)
plt.show()

### Positive support across two cohort calendars
The table counts candidate outcomes by episode establishment quarter. In the heatmap, colour
represents the number of positive snapshots at the intersection of an establishment
quarter and an observation quarter. Grey cells have no eligible observations. An eligible
cell with zero positives is shown at the bottom of the colour scale. Late establishment
cohorts contribute fewer observable snapshots before extraction. Use the heatmap for
support and timing, without interpreting colour as cohort churn risk.

Across all candidates, 18.24% are ineligible and 4.80% are right-censored. Their counts inform the usable observation volume, while 1.05% uses only the confirmed binary labels as its denominator.

In [ ]:
cohort_lookup = lifecycle.set_index("episode")["establishment quarter"]
cohort_observations = pd.DataFrame(
    (
        (
            cohort_lookup[row.subscription_id],
            pd.Timestamp(row.observed_at.date()).to_period("Q"),
            row.outcome.state.value,
        )
        for rows in (data.labelled, data.right_censored, data.pending, data.ineligible)
        for row in rows
    ),
    columns=["establishment quarter", "observation quarter", "state"],
)
cohort_mix = pd.crosstab(
    cohort_observations["establishment quarter"], cohort_observations["state"]
).reindex(columns=outcome_counts.index, fill_value=0)
assert cohort_mix.sum().equals(outcome_counts)
display(cohort_mix)
eligible = cohort_observations[
    cohort_observations["state"].isin(["confirmed_positive", "confirmed_negative"])
]
support = pd.crosstab(eligible["establishment quarter"], eligible["observation quarter"]).reindex(
    index=cohort_mix.index,
    columns=sorted(cohort_observations["observation quarter"].unique()),
    fill_value=0,
)
positive_support = pd.crosstab(
    eligible.loc[eligible["state"].eq("confirmed_positive"), "establishment quarter"],
    eligible.loc[eligible["state"].eq("confirmed_positive"), "observation quarter"],
).reindex(index=support.index, columns=support.columns, fill_value=0)
assert positive_support.to_numpy().sum() == positive_total
fig, ax = plt.subplots(figsize=(12, 5), layout="constrained")
values = positive_support.where(support.gt(0))
palette = plt.get_cmap("Blues").with_extremes(bad="#e1e1e1")
mesh = ax.imshow(values.to_numpy(dtype=float), cmap=palette, aspect="auto", vmin=0)
ax.grid(False)
ax.set_xticks(
    range(len(values.columns)), labels=values.columns.astype(str), rotation=45, ha="right"
)
ax.set_yticks(range(len(values.index)), labels=values.index.astype(str))
ax.set(
    title="Fieldwise positive support by establishment and observation quarter",
    xlabel="Observation quarter",
    ylabel="Establishment quarter",
)
fig.colorbar(mesh, ax=ax, label="Confirmed positive snapshots")
plt.show()

## Forward split reconciliation

Compare the previously reviewed and previously scored boundaries. The scored partition has already informed the historical investigation and should be treated as consumed for later model selection. Results below are descriptive diagnostics. The existing split function enforces observation order, maturity and episode isolation.

**Questions.** Which part of the original failure comes from the observation calendar, and which comes from split exclusions? Both boundaries below reproduce recorded experiments. The exclusive upper bounds constrain outcome horizon and ascertainment. Observations of a group already assigned to an earlier partition are excluded from later partitions. Compare class counts and independent positive episodes before using percentages.


In [ ]:
from collections import Counter
from datetime import UTC, datetime

from nowa_customer_intelligence.data.synthetic.modelling_dataset import (
    SplitBoundaries,
    split_modelling_dataset,
)

windows = {
    "Original": (datetime(2022, 7, 1, tzinfo=UTC), datetime(2023, 7, 1, tzinfo=UTC)),
    "Previously scored": (datetime(2023, 4, 1, tzinfo=UTC), datetime(2023, 9, 1, tzinfo=UTC)),
}
split_counts = []
exclusion_counts = []
selected_rows = []
episode_cohorts = lifecycle.set_index("episode")["establishment quarter"]
for name, (train_end, validation_end) in windows.items():
    test_end = datetime(2024, 4, 1, tzinfo=UTC)
    split = split_modelling_dataset(data, SplitBoundaries(train_end, validation_end, test_end))
    partition_ids = []
    partition_size = 0
    for partition, limit in zip(
        ("training", "validation", "test"), (train_end, validation_end, test_end), strict=True
    ):
        rows = getattr(split, partition)
        positives = [row for row in rows if row.outcome.binary_target == 1]
        partition_ids.append({row.subscription_id for row in rows})
        partition_size += len(rows)
        assert all(max(row.horizon_end, row.ascertainment_cutoff) < limit for row in rows)
        split_counts.append(
            {
                "split": name,
                "partition": partition,
                "rows": len(rows),
                "positives": len(positives),
                "prevalence (%)": 100 * len(positives) / len(rows),
                "positive episodes": len({row.subscription_id for row in positives}),
                "first observation": min(row.observed_at for row in rows).date(),
                "last observation": max(row.observed_at for row in rows).date(),
                "latest maturity": max(
                    max(row.horizon_end, row.ascertainment_cutoff) for row in rows
                ).date(),
            }
        )
        if name == "Previously scored":
            selected_rows.extend((partition, row) for row in rows)
    assert sum(map(len, partition_ids)) == len(set.union(*partition_ids))
    assert partition_size + len(split.excluded) == len(data.labelled)
    reasons = Counter(row.reason for row in split.excluded)
    positive_reasons = Counter(
        row.reason for row in split.excluded if row.observation.outcome.binary_target == 1
    )
    exclusion_counts.extend(
        {"split": name, "reason": reason, "rows": count, "positives": positive_reasons[reason]}
        for reason, count in reasons.items()
    )
    if name == "Previously scored":
        selected_rows.extend(
            (f"Excluded: {item.reason}", item.observation) for item in split.excluded
        )
display(pd.DataFrame(split_counts).round(2))
display(pd.DataFrame(exclusion_counts))

## Descriptive covariate and selection diagnostics

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "group": group,
            "establishment quarter": episode_cohorts[row.subscription_id],
            "positive": row.outcome.binary_target,
            "age (days)": row.features.predictors.episode_age_days,
            "plan": row.features.predictors.plan_id,
            "monthly recurring GBP": float(row.features.predictors.recurring_revenue),
            "mean active users": row.features.predictors.active_users_mean,
            "sessions": row.features.predictors.sessions_total,
            "observed usage months": row.features.predictors.observed_usage_months,
            "expected usage months": row.features.predictors.expected_usage_months,
            "usage coverage": row.features.predictors.usage_coverage_complete,
        }
        for group, row in selected_rows
    ]
)
metrics = [
    "age (days)",
    "monthly recurring GBP",
    "mean active users",
    "sessions",
    "observed usage months",
    "expected usage months",
]
display(comparison.groupby("group")[metrics].quantile([0.25, 0.5, 0.75]).round(1))
display(
    comparison.groupby("group")
    .agg(
        complete_usage_share=("usage coverage", "mean"),
        missing_active_users_share=("mean active users", lambda values: values.isna().mean()),
        positive_share=("positive", "mean"),
    )
    .round(4)
)
display(pd.crosstab(comparison["group"], comparison["plan"], normalize="index").round(3))
shares = pd.crosstab(comparison["establishment quarter"], comparison["group"], normalize="columns")
fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
shares[["training", "validation", "test"]].mul(100).plot(ax=ax, marker="o", markersize=3)
ax.set(
    title="Fieldwise previously scored split establishment cohorts",
    xlabel="Episode establishment quarter",
    ylabel="Partition observations (%)",
)
plt.show()

### Consequences for temporal evaluation
The left panel compares positive snapshot support with an identical partition order for both
recorded designs. The right panel shows the observation-level age distribution of assigned
episodes. Earlier-group exclusions favour a different mix of episode ages in later periods.
Several observations can refer to the same episode, so the boxes are row-weighted.

In [ ]:
partitions = ["training", "validation", "test"]
split_frame = pd.DataFrame(split_counts)
split_frame["partition"] = pd.Categorical(
    split_frame["partition"], categories=partitions, ordered=True
)
partition_positives = split_frame.pivot(index="partition", columns="split", values="positives")
age_samples = [comparison.loc[comparison["group"].eq(group), "age (days)"] for group in partitions]
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")
partition_positives.plot.bar(ax=left, color=["#23667d", "#db8351"], rot=0)
left.set(
    title="Fieldwise positive observations in existing splits",
    xlabel="Partition",
    ylabel="Confirmed positive snapshots",
)
right.boxplot(age_samples, tick_labels=[group.capitalize() for group in partitions])
right.set(
    title="Fieldwise episode age at observation in the previously scored split",
    xlabel="Partition",
    ylabel="Episode age (days)",
)
plt.show()

## Interpretation and implications

The original training period ends before the first eligible positive. The generator schedules selected terminations partway through the remaining simulated history and then creates distinct successor episodes. That design is consistent with the later concentration of events. Horizon crossings and assignment to an earlier episode group remove additional positives. The documented counts separate those split effects from observation ineligibility and censoring.

Batch 09 reported zero true positives among 19 held-out positive episodes. Late cohort concentration, differences in episode age and group selection are plausible contributors, alongside small positive support and the fictional data mechanism. The complete-usage flag is false across all compared groups even though mean active users is observed. Check whether month counts and feature availability convey distinct aspects of coverage before using either as a quality filter. The tables describe association and do not isolate causes. Rates and coverage are row-weighted, with repeated observations on many episodes.

For Phase 06, treat the scored period as development evidence and require a newly untouched evaluation scope before model selection. Predefine maturity, minimum independent positive support, grouping and evaluation rules. Check cohort representativeness and missingness drift on fresh evidence. This notebook did not fit models, optimise thresholds or choose replacement splits.

The 151 positives among 14,363 eligible observations give 1.05% prevalence. Accuracy can therefore look high with zero recall. Precision is sensitive to both false positives and the prevalence of the evaluated period. In the previously scored period, one positive episode changes recall by about 5.3 percentage points. Average Precision summarises the precision–recall curve and should be read alongside the period’s prevalence, the constant-score reference, and independent positive support. Future temporal evaluation should report precision, recall and Average Precision with the class counts and uncertainty appropriate to repeated observations. Predefine maturity and support requirements because early windows may contain no positives.

**Decision boundary.** The reference data can exercise point-in-time feature and outcome checks. The small, late positive signal and the use of one fictional plan limit discrimination studies. The split plots show that changing the observation window changes the population being compared. The age shift, earlier-episode exclusion and incomplete usage are plausible contributors to the previous outcome. The notebook does not separate their effects from the generator mechanics. Before a later modelling phase, specify a fresh holdout and a minimum number of independent positive episodes, then check age, coverage and cohort drift under the preregistered split.
